# 04 -- Session and news analysis

Paired scripts: `analysis/join_news_events.py` for the NEWS half (independently recomputes
`NEWS_BLACKOUT` status per `NewsManager.mqh`/section 10 for every journal decision, directly
useful given every real journal record's `news_state` is currently always empty -- the live
EA never sets it, see `analysis/schema.py`'s docstring), and `analysis/performance_breakdown.py`
for the time-of-day, session, mode, and news OUTCOME breakdowns.

**Fixed, 2026-07-22 Codex review finding (third round):** this notebook's own text
previously described an Asia/London/New-York UTC-hour session convention that no longer
exists in the code below -- the "Time-of-day breakdown" cell only ever groups by the real,
DERIVED `hour_of_day` dimension (see that cell's own docstring for why no invented
session-bucket substitute is used there).

**Corrected, 2026-07-22 Codex review finding (fourth round): this notebook previously
stopped at hour-of-day and news-window-membership, and explicitly claimed it did NOT
perform a session/mode/news OUTCOME breakdown at all** -- despite
`performance_breakdown.py`'s `OPTIONAL_DIMENSIONS` already supporting `session_state`,
`intraday_mode`, `news_state`, and `in_news_blackout`. The "Session / mode / news OUTCOME
breakdown" section below now actually reports win rate/expectancy by all four, on
clearly-labelled SYNTHETIC data (per reproducibility rule 7) -- the breakdown LOGIC does not
need to wait for real broker-session data, only the real-data VALUES do (see the closing
cell for what's still needed to run this same breakdown for real).

**Uses clearly-labelled SYNTHETIC journal/news/trade fixtures.** Real-data run: PENDING.

In [ ]:
import json
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.join_news_events import run as run_news_join

## News-blackout join

In [ ]:
tmp_dir = Path(tempfile.mkdtemp(prefix="themba_news_demo_"))

decision = {
    "signal_id": "sig-1",
    "timestamp_utc": "2026-07-21T14:05:30Z",
    "symbol": "XAUUSD",
    "market_family": "METAL",
    "intraday_mode": "SCALP",
    "regime": "REGIME_TRENDING_UP",
    "regime_confidence": 72.5,
    "direction": "BUY",
    "strategy": "TrendFollowingStrategy",
    "setup": "TrendlinePullback",
    "candlestick_pattern": None,
    "chart_pattern": None,
    "score": 68.0,
    "score_breakdown": {},
    "entry": 2350.55,
    "stop": 2345.10,
    "targets": [2361.45],
    "risk_percent": 0.3,
    "news_state": "",
    "session_state": "",
    "reasons_passed": [],
    "reasons_rejected": [],
    "ea_version": "1.01",
    "git_commit": "abc",
}
(tmp_dir / "decisions_20260721.jsonl").write_text(json.dumps(decision) + "\n", encoding="utf-8")

pd.DataFrame(
    [
        {
            "event_id": "e-nfp",
            "event_name": "NFP",
            "currency": "USD",
            "importance": 2,
            "scheduled_utc": "2026-07-21T14:10:00Z",  # 4m30s after the decision -- inside the window
        }
    ]
).to_csv(tmp_dir / "news.csv", index=False)

news_result = run_news_join(
    tmp_dir,
    tmp_dir / "news.csv",
    currency="USD",
    before_minutes=15,
    after_minutes=15,
    min_importance=2,
    repo_path=PROJECT_ROOT.parents[1],
)

print(f"n_decisions      = {news_result.n_decisions}")
print(f"n_in_blackout    = {news_result.n_in_blackout}")

assert news_result.n_in_blackout == 1
assert news_result.joined.iloc[0]["triggering_event_id"] == "e-nfp"

## Time-of-day breakdown (real, derived dimensions -- not an invented session bucket)

**Fixed, 2026-07-22 Codex review finding:** this cell previously defined a made-up
fixed-UTC-hour `session_for_hour` Asia/London/New-York bucketing function, computed inline
in the notebook rather than in a paired script -- and that bucketing was NOT sourced from
this project's actual session logic at all. `SessionManager.mqh` has **no fixed Asia/
London/New-York UTC-hour concept anywhere** -- it only ever computes session-time-remaining
from the BROKER'S OWN per-symbol session calendar (`SymbolInfoSessionTrade`, a live-MT5 API
this Python layer cannot call offline). Inventing a fixed-UTC-hour substitute was not
"porting the broker-session logic" as the master prompt requires; it was fabricating a
different, un-validated one.

This cell instead uses the real, paired `analysis/performance_breakdown.py` pipeline's
`hour_of_day`/`day_of_week` dimensions -- genuinely DERIVED from each decision's own
`entry_time` (no invented categorization). A true SESSION breakdown (Asia/London/NY-style)
would require either porting `SessionManager.mqh`'s real broker-session-table logic (needs a
real exported session table -- see `TASK-037_MT5_EXPORT_BRIDGE.md`) or consuming the
journal schema's own `session_state` field once the live EA populates it (see
`TASK-036_JOURNAL_PRODUCER_COMPLETION.md`) -- neither of which this notebook fabricates a
substitute for.

In [ ]:
import tempfile

from analysis.performance_breakdown import run as run_breakdown

# Trades entering at the same real UTC hour, three per hour, so the
# hour_of_day grouping below has genuine multi-trade statistics --
# hour_of_day is DERIVED from entry_time, not an invented session label.
synthetic_trades = pd.DataFrame(
    {
        "trade_id": [f"s{i}" for i in range(9)],
        "entry_time": (
            ["2026-07-21T02:00:00Z"] * 3
            + ["2026-07-21T09:00:00Z"] * 3
            + ["2026-07-22T15:00:00Z"] * 3
        ),
        "profit": [10.0, -5.0, 10.0, 10.0, 10.0, -5.0, -5.0, -5.0, 10.0],
    }
)
breakdown_dir = Path(tempfile.mkdtemp(prefix="themba_hourofday_demo_"))
trades_csv = breakdown_dir / "trades.csv"
synthetic_trades.to_csv(trades_csv, index=False)

performance_by_hour = run_breakdown(trades_csv, ["hour_of_day"])
print(performance_by_hour[["hour_of_day", "n_trades", "win_rate", "expectancy_dollars"]])

assert len(performance_by_hour) == 3
hour2_row = performance_by_hour[performance_by_hour["hour_of_day"] == 2].iloc[0]
assert abs(hour2_row["win_rate"] - (2.0 / 3.0)) < 1e-9  # 2 wins of 3
hour15_row = performance_by_hour[performance_by_hour["hour_of_day"] == 15].iloc[0]
assert abs(hour15_row["win_rate"] - (1.0 / 3.0)) < 1e-9  # 1 win of 3

## Session / mode / news OUTCOME breakdown (synthetic, added 2026-07-22 Codex review finding, fourth round)

**This notebook previously stopped short of the required deliverable.** The time-of-day
cell above proves `hour_of_day`/`day_of_week` grouping works; the news cell above proves
blackout-window detection works -- but neither ever reported actual trade OUTCOMES
(win rate, expectancy) broken down by `session_state`, `intraday_mode`, `news_state`, or
`in_news_blackout`, despite `performance_breakdown.py`'s `OPTIONAL_DIMENSIONS` already
supporting all four (see that module -- the gap was here, not there).

**Why this can be done now, on synthetic data, without waiting for real broker-session
data:** the reproducibility contract's own rule 7 is "if real evidence data is unavailable,
tests use clearly labelled synthetic fixtures and mark the real-data run pending" -- exactly
what every other notebook in this suite already does. There is no need to wait for a real
`SessionManager.mqh` broker-session export to prove the BREAKDOWN LOGIC itself is correct;
only the real-data `session_state` VALUES depend on that (see the closing markdown cell for
the producer/export mapping that still needs to exist before this can run on real data).

**session_state bucket convention used below** (synthetic, but a real, stated definition --
not an invented session-hour substitute like the old bug this notebook already fixed once):
`SessionManager.mqh`'s `SN_GetSessionMinutesRemaining` returns a continuous `remaining_ratio`
in `[0, 1]` (fraction of today's session remaining), never a 3-bucket label. This notebook
(and `TASK-036_JOURNAL_PRODUCER_COMPLETION.md`, which owns the real MQL5-side population)
now defines the mapping explicitly: `ratio >= 0.5` -> `"OPEN"`, `0.1 <= ratio < 0.5` ->
`"CLOSING_SOON"`, `ratio < 0.1` (or no session today) -> `"CLOSED"`.

In [ ]:
from analysis.performance_breakdown import run as run_breakdown

# Hand-traceable synthetic unified trades: 8 rows spanning both
# session_state buckets, both intraday_mode values, and both a
# news-quiet and a news-blackout news_state -- chosen so every
# breakdown below has a clean, hand-computable answer.
synthetic_unified_trades = pd.DataFrame(
    [
        {
            "profit": 10.0,
            "session_state": "OPEN",
            "intraday_mode": "SCALP",
            "news_state": "NONE",
            "in_news_blackout": False,
        },
        {
            "profit": 10.0,
            "session_state": "OPEN",
            "intraday_mode": "SCALP",
            "news_state": "NONE",
            "in_news_blackout": False,
        },
        {
            "profit": -5.0,
            "session_state": "OPEN",
            "intraday_mode": "DAY_TRADE",
            "news_state": "NONE",
            "in_news_blackout": False,
        },
        {
            "profit": 10.0,
            "session_state": "OPEN",
            "intraday_mode": "DAY_TRADE",
            "news_state": "NONE",
            "in_news_blackout": False,
        },
        {
            "profit": -5.0,
            "session_state": "CLOSING_SOON",
            "intraday_mode": "SCALP",
            "news_state": "NFP_NEARBY",
            "in_news_blackout": True,
        },
        {
            "profit": -5.0,
            "session_state": "CLOSING_SOON",
            "intraday_mode": "SCALP",
            "news_state": "NFP_NEARBY",
            "in_news_blackout": True,
        },
        {
            "profit": -5.0,
            "session_state": "CLOSING_SOON",
            "intraday_mode": "DAY_TRADE",
            "news_state": "NFP_NEARBY",
            "in_news_blackout": True,
        },
        {
            "profit": 10.0,
            "session_state": "CLOSING_SOON",
            "intraday_mode": "DAY_TRADE",
            "news_state": "NONE",
            "in_news_blackout": False,
        },
    ]
)
synthetic_unified_trades.insert(
    0, "trade_id", [f"su{i}" for i in range(len(synthetic_unified_trades))]
)

session_mode_news_dir = Path(tempfile.mkdtemp(prefix="themba_session_mode_news_demo_"))
unified_csv = session_mode_news_dir / "unified_trades.csv"
synthetic_unified_trades.to_csv(unified_csv, index=False)

by_session = run_breakdown(unified_csv, ["session_state"])
by_mode = run_breakdown(unified_csv, ["intraday_mode"])
by_news_state = run_breakdown(unified_csv, ["news_state"])
by_blackout = run_breakdown(unified_csv, ["in_news_blackout"])

print("-- by session_state --")
print(by_session[["session_state", "n_trades", "win_rate", "expectancy_dollars"]])
print("\n-- by intraday_mode --")
print(by_mode[["intraday_mode", "n_trades", "win_rate", "expectancy_dollars"]])
print("\n-- by news_state --")
print(by_news_state[["news_state", "n_trades", "win_rate", "expectancy_dollars"]])
print("\n-- by in_news_blackout --")
print(by_blackout[["in_news_blackout", "n_trades", "win_rate", "expectancy_dollars"]])

# Hand-computed: OPEN = [10,10,-5,10] -> n=4, win_rate=0.75, expectancy=6.25;
# CLOSING_SOON = [-5,-5,-5,10] -> n=4, win_rate=0.25, expectancy=-1.25.
open_row = by_session[by_session["session_state"] == "OPEN"].iloc[0]
assert open_row["n_trades"] == 4
assert abs(open_row["win_rate"] - 0.75) < 1e-9
assert abs(open_row["expectancy_dollars"] - 6.25) < 1e-9
closing_row = by_session[by_session["session_state"] == "CLOSING_SOON"].iloc[0]
assert abs(closing_row["win_rate"] - 0.25) < 1e-9
assert abs(closing_row["expectancy_dollars"] - (-1.25)) < 1e-9

# Hand-computed: NFP_NEARBY (== in_news_blackout True) = [-5,-5,-5] ->
# n=3, win_rate=0.0, expectancy=-5.0; NONE (== in_news_blackout False) =
# [10,10,-5,10,10] -> n=5, win_rate=0.8, expectancy=7.0.
nfp_row = by_news_state[by_news_state["news_state"] == "NFP_NEARBY"].iloc[0]
assert nfp_row["n_trades"] == 3
assert abs(nfp_row["win_rate"] - 0.0) < 1e-9
assert abs(nfp_row["expectancy_dollars"] - (-5.0)) < 1e-9
blackout_true_row = by_blackout[by_blackout["in_news_blackout"] == True].iloc[0]  # noqa: E712
assert blackout_true_row["n_trades"] == 3
assert abs(blackout_true_row["expectancy_dollars"] - (-5.0)) < 1e-9

## Real-data run: PENDING

Requires a real journal (batched runtime verification, TASK-025+), a real news-event export,
and enough real decisions spanning multiple hours -- none exist yet. A genuine SESSION
breakdown (distinct from this notebook's real, but coarser, hour-of-day breakdown) needs
either `SessionManager.mqh`'s own broker-session-table logic ported to Python (needs a real
exported session table -- see `TASK-037_MT5_EXPORT_BRIDGE.md`) or the journal schema's own
`session_state` field once the live EA populates it (see
`TASK-036_JOURNAL_PRODUCER_COMPLETION.md`, which now defines the exact
`OPEN`/`CLOSING_SOON`/`CLOSED` bucket-threshold mapping this notebook's synthetic section
above uses).

**Added, 2026-07-22 Codex review finding (fourth round):** the synthetic session/mode/news
OUTCOME breakdown section above already proves the breakdown mechanism itself is correct;
what remains genuinely pending is real data flowing through the SAME mechanism, which needs:
(1) `TASK-036`'s MQL5-side population of `intraday_mode`/`news_state`/`session_state` on
real journal decisions, using its now-defined bucket thresholds for `session_state`; (2)
`TASK-037`'s real trade-history/news-calendar exports so `join_signal_to_outcome.py` and
`join_news_events.py` have real input; and (3) actually running
`analysis/performance_breakdown.py` against that real unified CSV once both exist -- no new
Python-side capability is needed beyond what already exists and is tested here.